Lab 5

Goal: implement numerical methods for ordinary differential equations and verify accuracy by the double recalculation method.

Variant 21

Equation 1:
`y' = 2 + 1.2 y sin(x) - y^2`

Equation 2:
`y'' = cos(x + y) + 2.5 (x - y)`

Initial conditions used in the calculations:
`a = 0`, `b = 0.5`, `y(0) = 0`, and for the second equation `y'(0) = 1`.

Required accuracy: `0.001`.


In [1]:
import math
import numpy as np

A = 0.0
B = 0.5
EPS = 0.001
Y0 = 0.0
DY0 = 1.0


def equation1(x, y):
    return 2 + 1.2 * y * math.sin(x) - y * y


def equation2_rhs(x, y, z):
    return math.cos(x + y) + 2.5 * (x - y)


def euler_cauchy_method(n):
    h = (B - A) / (n - 1)
    x = np.linspace(A, B, n)
    y = np.zeros(n)
    y[0] = Y0
    for i in range(n - 1):
        predictor = y[i] + h * equation1(x[i], y[i])
        y[i + 1] = y[i] + h * (equation1(x[i], y[i]) + equation1(x[i + 1], predictor)) / 2
    return x, y


def runge_kutta_4_method(n):
    h = (B - A) / (n - 1)
    x = np.linspace(A, B, n)
    y = np.zeros(n)
    y[0] = Y0
    for i in range(n - 1):
        xi = x[i]
        yi = y[i]
        k1 = h * equation1(xi, yi)
        k2 = h * equation1(xi + h / 2, yi + k1 / 2)
        k3 = h * equation1(xi + h / 2, yi + k2 / 2)
        k4 = h * equation1(xi + h, yi + k3)
        y[i + 1] = yi + (k1 + 2 * k2 + 2 * k3 + k4) / 6
    return x, y


def rk4_step_for_system(x, y, z, h):
    k1y = h * z
    k1z = h * equation2_rhs(x, y, z)

    k2y = h * (z + k1z / 2)
    k2z = h * equation2_rhs(x + h / 2, y + k1y / 2, z + k1z / 2)

    k3y = h * (z + k2z / 2)
    k3z = h * equation2_rhs(x + h / 2, y + k2y / 2, z + k2z / 2)

    k4y = h * (z + k3z)
    k4z = h * equation2_rhs(x + h, y + k3y, z + k3z)

    next_y = y + (k1y + 2 * k2y + 2 * k3y + k4y) / 6
    next_z = z + (k1z + 2 * k2z + 2 * k3z + k4z) / 6
    return next_y, next_z


def adams_method(order, n):
    h = (B - A) / (n - 1)
    x = np.linspace(A, B, n)
    y = np.zeros(n)
    z = np.zeros(n)
    y[0] = Y0
    z[0] = DY0

    for i in range(order - 1):
        y[i + 1], z[i + 1] = rk4_step_for_system(x[i], y[i], z[i], h)

    for i in range(order - 1, n - 1):
        if order == 3:
            y[i + 1] = y[i] + h * (23 * z[i] - 16 * z[i - 1] + 5 * z[i - 2]) / 12
            fi = equation2_rhs(x[i], y[i], z[i])
            fim1 = equation2_rhs(x[i - 1], y[i - 1], z[i - 1])
            fim2 = equation2_rhs(x[i - 2], y[i - 2], z[i - 2])
            z[i + 1] = z[i] + h * (23 * fi - 16 * fim1 + 5 * fim2) / 12
        else:
            y[i + 1] = y[i] + h * (55 * z[i] - 59 * z[i - 1] + 37 * z[i - 2] - 9 * z[i - 3]) / 24
            fi = equation2_rhs(x[i], y[i], z[i])
            fim1 = equation2_rhs(x[i - 1], y[i - 1], z[i - 1])
            fim2 = equation2_rhs(x[i - 2], y[i - 2], z[i - 2])
            fim3 = equation2_rhs(x[i - 3], y[i - 3], z[i - 3])
            z[i + 1] = z[i] + h * (55 * fi - 59 * fim1 + 37 * fim2 - 9 * fim3) / 24

    return x, y, z


def solve_with_double_recalculation(method):
    n = 8
    previous = None
    while True:
        current = method(n)
        if previous is not None:
            previous_y = previous[1]
            current_y = current[1]
            max_diff = max(abs(previous_y[i] - current_y[2 * i]) for i in range(len(previous_y)))
            if max_diff < EPS:
                return previous, (*current, n), max_diff
        previous = (*current, n)
        n *= 2


def build_last_points_table(previous, current, count_last=16):
    x_prev, y_prev, *_, _ = previous
    x_curr, y_curr, *_, _ = current
    rows = []
    for i in range(len(x_curr) - count_last, len(x_curr)):
        if i % 2 == 0:
            j = i // 2
            rows.append((x_curr[i], y_prev[j], y_curr[i], abs(y_curr[i] - y_prev[j])))
        else:
            rows.append((x_curr[i], None, y_curr[i], None))
    return rows


def build_prev_iteration_table(previous, current, count_last=8):
    x_prev, y_prev, *_, _ = previous
    x_curr, y_curr, *_, _ = current
    return [(x_prev[i], y_prev[i], y_curr[2 * i], abs(y_curr[2 * i] - y_prev[i])) for i in range(len(x_prev) - count_last, len(x_prev))]


def fmt(value):
    return '-' if value is None else f'{value:.6f}'


def print_table(rows):
    print(f"{'x_k':>10} | {'y_prev':>14} | {'y_curr':>14} | {'abs diff':>10}")
    print('-' * 60)
    for row in rows:
        print(' | '.join(f'{fmt(v):>14}' if idx else f'{fmt(v):>10}' for idx, v in enumerate(row)))


In [2]:
results = []
methods = [
    ('Equation 1. Euler-Cauchy method', lambda n: euler_cauchy_method(n)),
    ('Equation 1. Runge-Kutta 4 method', lambda n: runge_kutta_4_method(n)),
    ('Equation 2. Adams 3rd-order method', lambda n: adams_method(3, n)),
    ('Equation 2. Adams 4th-order method', lambda n: adams_method(4, n)),
]

for title, method in methods:
    previous, current, max_diff = solve_with_double_recalculation(method)
    results.append({
        'title': title,
        'previous': previous,
        'current': current,
        'max_diff': max_diff,
        'last16': build_last_points_table(previous, current),
        'prev8': build_prev_iteration_table(previous, current),
    })


In [3]:
print("Initial data: a = 0, b = 0.5, y(0) = 0, and for equation 2 also y'(0) = 1.")
print("Equation 1: y' = 2 + 1.2 y sin(x) - y^2")
print("Equation 2: y'' = cos(x + y) + 2.5 (x - y)")
print()

for item in results:
    print(item['title'])
    print(f"Previous iteration: n = {item['previous'][-1]}")
    print(f"Final iteration: n = {item['current'][-1]}")
    print(f"Maximum difference in common nodes: {item['max_diff']:.6f}")
    print('Last 16 points of the final iteration:')
    print_table(item['last16'])
    print('Last 8 points of the previous iteration:')
    print_table(item['prev8'])
    print()


Initial data: a = 0, b = 0.5, y(0) = 0, and for equation 2 also y'(0) = 1.
Equation 1: y' = 2 + 1.2 y sin(x) - y^2
Equation 2: y'' = cos(x + y) + 2.5 (x - y)

Equation 1. Euler-Cauchy method
Previous iteration: n = 512
Final iteration: n = 1024
Maximum difference in common nodes: 0.000810
Last 16 points of the final iteration:
       x_k |         y_prev |         y_curr |   abs diff
------------------------------------------------------------
  0.492669 |       0.928019 |       0.927216 |       0.000803
  0.493157 |              - |       0.928030 |              -
  0.493646 |       0.929648 |       0.928844 |       0.000804
  0.494135 |              - |       0.929658 |              -
  0.494624 |       0.931277 |       0.930472 |       0.000805
  0.495112 |              - |       0.931285 |              -
  0.495601 |       0.932904 |       0.932098 |       0.000806
  0.496090 |              - |       0.932911 |              -
  0.496579 |       0.934530 |       0.933723 |       0.0